In [1]:
import os
import json
from pathlib import Path
import datetime as dt
import re

import pandas as pd
import numpy as np
from PIL import Image
from pathlib import PureWindowsPath
from tqdm.notebook import tqdm 

import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from PIL import Image
import skimage.io

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

import skimage, torch, torchvision
import torchxrayvision as xrv
import torch.nn.functional as F
import torchvision.transforms as transforms
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.optim.lr_scheduler as lr_scheduler
import torch.optim as optim

from transformers import AutoModel, AutoImageProcessor

current_path = os.getcwd()

os.chdir('..')
print(f"current path: {current_path}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

current path: /home/DAHS1/gangmin/my_research/clinical_multimodal_learning/preprocess


# 1. Segmentation (CXLSeg Dataset)

In [2]:
seg_meta = pd.read_csv("./data/segment_mask/physionet.org/files/chest-x-ray-segmentation/1.0.0/CXLSeg-metadata.csv")
seg_meta.head()

,dicom_id,subject_id,study_id,Reports,ViewPosition,PerformedProcedureStepDescription,Rows,Columns,StudyDate,StudyTime,ProcedureCodeSequence_CodeMeaning,ViewCodeSequence_CodeMeaning,PatientOrientationCodeSequence_CodeMeaning
0,02aa804e-bde0afdd-112c0b34-7bc16630-4e384014,10000032,50414267,No acute cardiopulmonary process.,PA,CHEST (PA AND LAT),3056,2544,21800506,213014.531,CHEST (PA AND LAT),postero-anterior,Erect
1,2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab,10000032,53189527,No acute cardiopulmonary abnormality.,PA,CHEST (PA AND LAT),3056,2544,21800626,165500.312,CHEST (PA AND LAT),postero-anterior,Erect
2,68b5c4b1-227d0485-9cc38c3f-7b84ab51-4b472714,10000032,53911762,No acute intrathoracic process.,AP,CHEST (PORTABLE AP),2705,2539,21800723,80556.875,CHEST (PORTABLE AP),antero-posterior,NaN
3,fffabebf-74fd3a1f-673b6b41-96ec0ac9-2ab69818,10000032,53911762,No acute intrathoracic process.,AP,CHEST (PORTABLE AP),2906,2258,21800723,80556.875,CHEST (PORTABLE AP),antero-posterior,Erect
4,ea030e7a-2e3b1346-bc518786-7a8fd698-f673b44c,10000032,56699142,No acute cardiopulmonary process.,AP,CHEST (PORTABLE AP),3056,2544,21800805,234424.765,CHEST (PORTABLE AP),antero-posterior,NaN


In [3]:
seg_mask = pd.read_csv("./data/segment_mask/physionet.org/files/chest-x-ray-segmentation/1.0.0/CXLSeg-mask.csv")
seg_mask.head()

,dicom_id,subject_id,study_id,DicomPath
0,02aa804e-bde0afdd-112c0b34-7bc16630-4e384014,10000032,50414267,files/p10/p10000032/s50414267/02aa804e-bde0afd...
1,2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab,10000032,53189527,files/p10/p10000032/s53189527/2a2277a9-b0ded15...
2,68b5c4b1-227d0485-9cc38c3f-7b84ab51-4b472714,10000032,53911762,files/p10/p10000032/s53911762/68b5c4b1-227d048...
3,fffabebf-74fd3a1f-673b6b41-96ec0ac9-2ab69818,10000032,53911762,files/p10/p10000032/s53911762/fffabebf-74fd3a1...
4,ea030e7a-2e3b1346-bc518786-7a8fd698-f673b44c,10000032,56699142,files/p10/p10000032/s56699142/ea030e7a-2e3b134...


In [4]:
seg_label = pd.read_csv("./data/segment_mask/physionet.org/files/chest-x-ray-segmentation/1.0.0/CXLSeg-segmented.csv")
seg_label.head()

,dicom_id,subject_id,study_id,DicomPath,Atelectasis,Cardiomegaly,Consolidation,Edema,Enlarged Cardiomediastinum,Fracture,Lung Lesion,Lung Opacity,No Finding,Pleural Effusion,Pleural Other,Pneumonia,Pneumothorax,Support Devices
0,02aa804e-bde0afdd-112c0b34-7bc16630-4e384014,10000032,50414267,files/p10/p10000032/s50414267/02aa804e-bde0afd...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
1,2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab,10000032,53189527,files/p10/p10000032/s53189527/2a2277a9-b0ded15...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
2,68b5c4b1-227d0485-9cc38c3f-7b84ab51-4b472714,10000032,53911762,files/p10/p10000032/s53911762/68b5c4b1-227d048...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
3,fffabebf-74fd3a1f-673b6b41-96ec0ac9-2ab69818,10000032,53911762,files/p10/p10000032/s53911762/fffabebf-74fd3a1...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
4,ea030e7a-2e3b1346-bc518786-7a8fd698-f673b44c,10000032,56699142,files/p10/p10000032/s56699142/ea030e7a-2e3b134...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN


In [5]:
seg_mask = seg_mask.rename(columns={'DicomPath': 'lung_mask_path'})

In [6]:
# 경로 처리
lung_data_path = '/home/DAHS1/gangmin/my_research/clinical_multimodal_learning/data/segment_mask/physionet.org/files/chest-x-ray-segmentation/1.0.0'

seg_mask['lung_mask_path'] = seg_mask['lung_mask_path'].apply(lambda x: os.path.join(lung_data_path, 'lung_mask', x) if x is not None else None)

In [7]:
seg_mask['lung_mask_path'][0]

'/home/DAHS1/gangmin/my_research/clinical_multimodal_learning/data/segment_mask/physionet.org/files/chest-x-ray-segmentation/1.0.0/lung_mask/files/p10/p10000032/s50414267/02aa804e-bde0afdd-112c0b34-7bc16630-4e384014-mask.jpg'

In [8]:
seg_mask

,dicom_id,subject_id,study_id,lung_mask_path
0,02aa804e-bde0afdd-112c0b34-7bc16630-4e384014,10000032,50414267,/home/DAHS1/gangmin/my_research/clinical_multi...
1,2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab,10000032,53189527,/home/DAHS1/gangmin/my_research/clinical_multi...
2,68b5c4b1-227d0485-9cc38c3f-7b84ab51-4b472714,10000032,53911762,/home/DAHS1/gangmin/my_research/clinical_multi...
3,fffabebf-74fd3a1f-673b6b41-96ec0ac9-2ab69818,10000032,53911762,/home/DAHS1/gangmin/my_research/clinical_multi...
4,ea030e7a-2e3b1346-bc518786-7a8fd698-f673b44c,10000032,56699142,/home/DAHS1/gangmin/my_research/clinical_multi...
...,...,...,...,...
243319,3fcd0406-9b111603-feae7033-96632b3a-111333e5,19999733,57132437,/home/DAHS1/gangmin/my_research/clinical_multi...
243320,428e2c18-5721d8f3-35a05001-36f3d080-9053b83c,19999733,57132437,/home/DAHS1/gangmin/my_research/clinical_multi...
243321,58766883-376a15ce-3b323a28-6af950a0-16b793bd,19999987,55368167,/home/DAHS1/gangmin/my_research/clinical_multi...
243322,7ba273af-3d290f8d-e28d0ab4-484b7a86-7fc12b08,19999987,58621812,/home/DAHS1/gangmin/my_research/clinical_multi...


# 2. Lesion (EXT-ILS Dataset)

In [9]:
# # physionet example code

# import json

# # --- Configuration ---
# SPLIT = 'train'

# JSON_PATH = './data/ext-ils/physionet.org/files/mimic-cxr-ext-ils/1.0.0/mimic_ils_instruction_answer.json'

# # --- Load Dataset ---
# with open(JSON_PATH, 'r') as f:
#     dataset = json.load(f)

# # --- Visualize Samples ---
# print(f"Successfully loaded dataset. Visualizing samples from '{SPLIT}' split...\n")

# for i, (study_id, data) in enumerate(dataset[SPLIT].items()):
#     if i >= 5:
#         break

#     print(f"{'='*20} Sample {i+1} (Study ID: {study_id}) {'='*20}")
#     print(f"Image Path: {data['image_path']}\n")

#     pairs_data = data['instruction_answer_pairs']

#     # Iterate over pair types to reduce code duplication
#     for pair_type in ['positive_pairs', 'negative_pairs']:
#         pairs = pairs_data.get(pair_type, [])
#         header = pair_type.replace('_', ' ').title()
#         print(f"[{header}] - {len(pairs)} item(s)")

#         for idx, pair in enumerate(pairs):
#             print(f"  {idx+1}. Instruction : {pair['instruction']}")
#             print(f"     Answer        : {pair['answer']}")
#             print(f"     Mask Path     : {pair['seg_mask_path']}")

#         print("-" * 3)

In [10]:
"""
추출이 필요한 key 목록
- image_path: 원본 X-ray 이미지 경로
- seg_mask_path: 병변 마스크 이미지 경로
- target: 병변의 종류 (Class Label - 예: pneumonia, effusion 등)
- grounded_location: 해당 병변이 실제 존재하는 해부학적 위치 정보
- seg: True인 경우만 마스크가 존재하므로, 학습 시 이 플래그를 기준으로 필터링
"""

JSON_PATH = './data/ext-ils/physionet.org/files/mimic-cxr-ext-ils/1.0.0/mimic_ils_instruction_answer.json'

def build_lesion_dataframe(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    
    target_lesions = ['cardiomegaly', 'pneumonia', 'atelectasis', 'opacity', 'consolidation', 'edema', 'effusion']
    rows = []
    
    for split in ['train', 'val', 'test']:
        if split not in raw_data:
            continue
            
        for study_id, info in raw_data[split].items():
            entry = {
                'study_id': study_id,
                'subject_id': info.get('subject_id'),
                'image_path': info.get('image_path'),
                'split': split,
                'report_content': info.get('section_content'),
            }
            
            for lesion in target_lesions:
                entry[f'label_{lesion}'] = 0
                entry[f'mask_{lesion}'] = None
                entry[f'loc_{lesion}'] = []
            
            pos_pairs = info.get('instruction_answer_pairs', {}).get('positive_pairs', [])

            for pair in pos_pairs:
                lesion_type = pair.get('target')
                if lesion_type in target_lesions:
                    entry[f'label_{lesion_type}'] = 1
                    entry[f'mask_{lesion_type}'] = pair.get('seg_mask_path')
                    entry[f'loc_{lesion_type}'] = pair.get('grounded_location', [])

            rows.append(entry)

    return pd.DataFrame(rows)

df_lesion = build_lesion_dataframe(JSON_PATH)

In [11]:
df_lesion.columns

Index(['study_id', 'subject_id', 'image_path', 'split', 'report_content',
       'label_cardiomegaly', 'mask_cardiomegaly', 'loc_cardiomegaly',
       'label_pneumonia', 'mask_pneumonia', 'loc_pneumonia',
       'label_atelectasis', 'mask_atelectasis', 'loc_atelectasis',
       'label_opacity', 'mask_opacity', 'loc_opacity', 'label_consolidation',
       'mask_consolidation', 'loc_consolidation', 'label_edema', 'mask_edema',
       'loc_edema', 'label_effusion', 'mask_effusion', 'loc_effusion'],
      dtype='object')

In [12]:
# 경로 처리
lesion_data_path = '/home/DAHS1/gangmin/my_research/clinical_multimodal_learning/data/ext-ils/physionet.org/files/mimic-cxr-ext-ils/1.0.0/'
mask_cols = [col for col in df_lesion.columns if col.startswith('mask_')]

for col in mask_cols:
    df_lesion[col] = df_lesion[col].apply(lambda x: os.path.join(lesion_data_path, 'lesion_mask', x) if x is not None else None)

# 대괄호 처리
loc_cols = [col for col in df_lesion.columns if col.startswith('loc_')]

for col in loc_cols:
    df_lesion[col] = df_lesion[col].apply(lambda x: ", ".join(x) if isinstance(x, list) and len(x) > 0 else np.nan)

# cardiomegaly의 위치 정보는 heart로 고정함.
df_lesion.loc[
    (df_lesion['label_cardiomegaly'] == 1) & (df_lesion['loc_cardiomegaly'].isna()), 
    'loc_cardiomegaly'
] = 'heart'

df_lesion = df_lesion.drop(columns=['report_content', 'split'])

df_lesion['dicom_id'] = df_lesion['image_path'].str.split('/').str[-1].str.replace('.jpg', '', regex=False)
df_lesion['image_path'] = 'files/' + df_lesion['image_path']

# 3. Radiology Report
- MIMIC-CXR DICOM provided txt files

In [13]:
def extract_sections(text, file_id):
    """
    Radiology report에서 FINDINGS와 IMPRESSION 섹션을 추출함.
    """
    findings_pattern = re.compile(r'FINDINGS:(.*?)((?:IMPRESSION:)|$)', re.DOTALL | re.IGNORECASE)
    impression_pattern = re.compile(r'IMPRESSION:(.*)', re.DOTALL | re.IGNORECASE)

    # Findings 추출
    findings = findings_pattern.search(text)

    # Impression 추출
    impression = impression_pattern.search(text)

    f_text = findings.group(1).strip() if findings else ""
    i_text = impression.group(1).strip() if impression else ""

    combined_text = f"{f_text} {i_text}".strip()

    if not combined_text:
        combined_text = "No FINDINGS or IMPRESSION"
    
    return combined_text


def build_report_dataframe(base_path):
    base_path = Path(base_path)
    rows = []
    
    p_folders = [f"p{i}" for i in range(10, 20)]
    
    for p_sub in p_folders:
        current_dir = base_path / p_sub
        if not current_dir.exists():
            continue
            
        print(f"\nProcessing {p_sub}...")
        txt_files = list(current_dir.rglob("*.txt"))

        for txt_path in tqdm(txt_files):
            study_id = txt_path.stem
            subject_id = txt_path.parent.name
            
            content = txt_path.read_text(encoding='utf-8')
            needed_sections = extract_sections(content, file_id=f"{subject_id}/{study_id}")
            
            rows.append({
                'subject_id': subject_id,
                'study_id': study_id,
                'report': needed_sections,
                'txt_path': str(txt_path)
            })

    return pd.DataFrame(rows)

REPORT_BASE_PATH = '/home/DAHS1/gangmin/my_research/clinical_multimodal_learning/data/reports/'
df_reports = build_report_dataframe(REPORT_BASE_PATH)


Processing p10...


  0%|          | 0/22197 [00:00<?, ?it/s]


Processing p11...


  0%|          | 0/23358 [00:00<?, ?it/s]


Processing p12...


...

  0%|          | 0/22695 [00:00<?, ?it/s]


Processing p18...


  0%|          | 0/22929 [00:00<?, ?it/s]


Processing p19...


  0%|          | 0/22830 [00:00<?, ?it/s]

In [14]:
df_reports

,subject_id,study_id,report,txt_path
0,p10992808,s56501444,Mild prominence of the cardiac silhouette is l...,/home/DAHS1/gangmin/my_research/clinical_multi...
1,p10509699,s57504847,"As compared to the previous radiograph, the lu...",/home/DAHS1/gangmin/my_research/clinical_multi...
2,p10509699,s54509601,"Compared to the previous radiograph, the lung ...",/home/DAHS1/gangmin/my_research/clinical_multi...
3,p10509699,s55033134,The lung volumes are low. Moderate cardiomega...,/home/DAHS1/gangmin/my_research/clinical_multi...
4,p10637554,s58966893,"The cardiac, mediastinal and hilar contours ap...",/home/DAHS1/gangmin/my_research/clinical_multi...
...,...,...,...,...
227830,p19422157,s57292946,Sternotomy wires appear intact and appropriate...,/home/DAHS1/gangmin/my_research/clinical_multi...
227831,p19171097,s51355653,No evidence of acute cardiopulmonary abnormali...,/home/DAHS1/gangmin/my_research/clinical_multi...
227832,p19171097,s58399526,Frontal and lateral views of the chest. Heart...,/home/DAHS1/gangmin/my_research/clinical_multi...
227833,p19250934,s50196037,"The cardiac, mediastinal and hilar contours ap...",/home/DAHS1/gangmin/my_research/clinical_multi...


# 4. Main CXR-JPG (MIMIC-CXR-JPG Dataset)

In [15]:
df_mimic_cxr_chexpert = pd.read_csv('/home/DAHS1/gangmin/my_research/data/mimic-cxr-2.0.0-chexpert.csv')
df_mimic_cxr_metadata = pd.read_csv('/home/DAHS1/gangmin/my_research/data/mimic-cxr-2.0.0-metadata.csv')

In [16]:
cxr_txt_path = pd.read_parquet("/home/DAHS1/gangmin/my_research/clinical_multimodal_learning/data/image_filenames.parquet")

# dicom_id 추출: 마지막 '/' 이후 문자열
cxr_txt_path['dicom_id'] = cxr_txt_path['file_path'].str.extract(r'/([^/]+)\.jpg$')
cxr_txt_path = cxr_txt_path.rename(columns={'file_path': 'image_path'})
cxr_txt_path.head()

,image_path,dicom_id
0,files/p10/p10000032/s50414267/02aa804e-bde0afd...,02aa804e-bde0afdd-112c0b34-7bc16630-4e384014
1,files/p10/p10000032/s50414267/174413ec-4ec4c1f...,174413ec-4ec4c1f7-34ea26b7-c5f994f8-79ef1962
2,files/p10/p10000032/s53189527/2a2277a9-b0ded15...,2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab
3,files/p10/p10000032/s53189527/e084de3b-be89b11...,e084de3b-be89b11e-20fe3f9f-9c8d8dfe-4cfd202c
4,files/p10/p10000032/s53911762/68b5c4b1-227d048...,68b5c4b1-227d0485-9cc38c3f-7b84ab51-4b472714


In [17]:
df_mimic_cxr_metadata = df_mimic_cxr_metadata.merge(cxr_txt_path, on='dicom_id', how='inner')

In [18]:
print(f"전체 이미지 수: {len(df_mimic_cxr_metadata)}")
df_mimic_cxr_metadata.head(1)

전체 이미지 수: 377110


,dicom_id,subject_id,study_id,PerformedProcedureStepDescription,ViewPosition,Rows,Columns,StudyDate,StudyTime,ProcedureCodeSequence_CodeMeaning,ViewCodeSequence_CodeMeaning,PatientOrientationCodeSequence_CodeMeaning,image_path
0,02aa804e-bde0afdd-112c0b34-7bc16630-4e384014,10000032,50414267,CHEST (PA AND LAT),PA,3056,2544,21800506,213014.531,CHEST (PA AND LAT),postero-anterior,Erect,files/p10/p10000032/s50414267/02aa804e-bde0afd...


In [19]:
df_mimic_cxr_metadata['ViewPosition'].value_counts()

ViewPosition
AP                147173
PA                 96161
LATERAL            82853
LL                 35133
PA LLD                 4
LAO                    3
RAO                    3
AP AXIAL               2
AP LLD                 2
XTABLE LATERAL         2
AP RLD                 2
SWIMMERS               1
PA RLD                 1
LPO                    1
Name: count, dtype: int64

In [20]:
df_mimic_cxr_metadata = df_mimic_cxr_metadata[(df_mimic_cxr_metadata['ViewPosition']=="AP") | (df_mimic_cxr_metadata['ViewPosition'] == "PA")].reset_index(drop=True)

# Uncertain to positive (Based on Chexpert paper)
df_mimic_cxr_chexpert = df_mimic_cxr_chexpert.fillna(0) # 불가피함.

meta_cols = ['subject_id', 'study_id']
target_cols = df_mimic_cxr_chexpert.columns.difference(meta_cols)
df_mimic_cxr_chexpert[target_cols] = df_mimic_cxr_chexpert[target_cols].replace(-1, 1)

In [21]:
total_jpg_df = df_mimic_cxr_metadata.merge(df_mimic_cxr_chexpert, on=['subject_id', 'study_id'], how='inner').reset_index(drop=True)

In [22]:
total_jpg_df['StudyDateForm'] = pd.to_datetime(total_jpg_df['StudyDate'], format='%Y%m%d')
total_jpg_df['StudyTimeForm'] = total_jpg_df.apply(lambda x : '%#010.3f' % x['StudyTime'] ,1)
total_jpg_df['StudyTimeForm'] = pd.to_datetime(total_jpg_df['StudyTimeForm'], format='%H%M%S.%f').dt.time
total_jpg_df['cxrtime'] = total_jpg_df.apply(lambda r : dt.datetime.combine(r['StudyDateForm'],r['StudyTimeForm']), 1)

In [23]:
total_jpg_df.columns

Index(['dicom_id', 'subject_id', 'study_id',
       'PerformedProcedureStepDescription', 'ViewPosition', 'Rows', 'Columns',
       'StudyDate', 'StudyTime', 'ProcedureCodeSequence_CodeMeaning',
       'ViewCodeSequence_CodeMeaning',
       'PatientOrientationCodeSequence_CodeMeaning', 'image_path',
       'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
       'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 'Lung Opacity',
       'No Finding', 'Pleural Effusion', 'Pleural Other', 'Pneumonia',
       'Pneumothorax', 'Support Devices', 'StudyDateForm', 'StudyTimeForm',
       'cxrtime'],
      dtype='object')

In [24]:
total_jpg_df = total_jpg_df[[
    'subject_id', 'study_id', 'dicom_id', 'image_path', 'ViewPosition', 'cxrtime', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Enlarged Cardiomediastinum', 
    'Fracture', 'Lung Lesion', 'Lung Opacity', 'No Finding', 'Pleural Effusion', 'Pleural Other', 'Pneumonia', 'Pneumothorax', 'Support Devices'
]]

In [25]:
total_jpg_df

,subject_id,study_id,dicom_id,image_path,ViewPosition,cxrtime,Atelectasis,Cardiomegaly,Consolidation,Edema,Enlarged Cardiomediastinum,Fracture,Lung Lesion,Lung Opacity,No Finding,Pleural Effusion,Pleural Other,Pneumonia,Pneumothorax,Support Devices
0,10000032,50414267,02aa804e-bde0afdd-112c0b34-7bc16630-4e384014,files/p10/p10000032/s50414267/02aa804e-bde0afd...,PA,2180-05-06 21:30:14.531,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,10000032,53189527,2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab,files/p10/p10000032/s53189527/2a2277a9-b0ded15...,PA,2180-06-26 16:55:00.312,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,10000032,53911762,68b5c4b1-227d0485-9cc38c3f-7b84ab51-4b472714,files/p10/p10000032/s53911762/68b5c4b1-227d048...,AP,2180-07-23 08:05:56.875,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,10000032,53911762,fffabebf-74fd3a1f-673b6b41-96ec0ac9-2ab69818,files/p10/p10000032/s53911762/fffabebf-74fd3a1...,AP,2180-07-23 08:05:56.875,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,10000032,56699142,ea030e7a-2e3b1346-bc518786-7a8fd698-f673b44c,files/p10/p10000032/s56699142/ea030e7a-2e3b134...,AP,2180-08-05 23:44:24.765,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
243319,19999733,57132437,3fcd0406-9b111603-feae7033-96632b3a-111333e5,files/p19/p19999733/s57132437/3fcd0406-9b11160...,PA,2152-07-08 22:45:50.171,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
243320,19999733,57132437,428e2c18-5721d8f3-35a05001-36f3d080-9053b83c,files/p19/p19999733/s57132437/428e2c18-5721d8f...,PA,2152-07-08 22:45:50.171,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
243321,19999987,55368167,58766883-376a15ce-3b323a28-6af950a0-16b793bd,files/p19/p19999987/s55368167/58766883-376a15c...,AP,2145-11-04 05:14:48.218,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
243322,19999987,58621812,7ba273af-3d290f8d-e28d0ab4-484b7a86-7fc12b08,files/p19/p19999987/s58621812/7ba273af-3d290f8...,AP,2145-11-02 20:28:09.234,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


# 데이터 결합

In [26]:
df_combined_1 = pd.merge(total_jpg_df, seg_mask, on=['subject_id', 'study_id', 'dicom_id'], how='left')

In [27]:
print(f"전체 행 수: {len(df_combined_1)}")
df_combined_1.head()

전체 행 수: 243324


,subject_id,study_id,dicom_id,image_path,ViewPosition,cxrtime,Atelectasis,Cardiomegaly,Consolidation,Edema,...,Fracture,Lung Lesion,Lung Opacity,No Finding,Pleural Effusion,Pleural Other,Pneumonia,Pneumothorax,Support Devices,lung_mask_path
0,10000032,50414267,02aa804e-bde0afdd-112c0b34-7bc16630-4e384014,files/p10/p10000032/s50414267/02aa804e-bde0afd...,PA,2180-05-06 21:30:14.531,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,/home/DAHS1/gangmin/my_research/clinical_multi...
1,10000032,53189527,2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab,files/p10/p10000032/s53189527/2a2277a9-b0ded15...,PA,2180-06-26 16:55:00.312,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,/home/DAHS1/gangmin/my_research/clinical_multi...
2,10000032,53911762,68b5c4b1-227d0485-9cc38c3f-7b84ab51-4b472714,files/p10/p10000032/s53911762/68b5c4b1-227d048...,AP,2180-07-23 08:05:56.875,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,/home/DAHS1/gangmin/my_research/clinical_multi...
3,10000032,53911762,fffabebf-74fd3a1f-673b6b41-96ec0ac9-2ab69818,files/p10/p10000032/s53911762/fffabebf-74fd3a1...,AP,2180-07-23 08:05:56.875,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,/home/DAHS1/gangmin/my_research/clinical_multi...
4,10000032,56699142,ea030e7a-2e3b1346-bc518786-7a8fd698-f673b44c,files/p10/p10000032/s56699142/ea030e7a-2e3b134...,AP,2180-08-05 23:44:24.765,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,/home/DAHS1/gangmin/my_research/clinical_multi...


In [28]:
df_combined_1['dicom_id'][0]

'02aa804e-bde0afdd-112c0b34-7bc16630-4e384014'

In [29]:
df_combined_1['subject_id'] = df_combined_1['subject_id'].astype(str)
df_lesion['subject_id'] = df_lesion['subject_id'].astype(str)

df_combined_1['study_id'] = df_combined_1['study_id'].astype(str)
df_lesion['study_id'] = df_lesion['study_id'].astype(str)

df_combined_1['dicom_id'] = df_combined_1['dicom_id'].astype(str)
df_lesion['dicom_id'] = df_lesion['dicom_id'].astype(str)

df_lesion['study_id'] = df_lesion['study_id'].str.lstrip('s')
df_lesion['subject_id'] = df_lesion['subject_id'].str.lstrip('p')

In [30]:
df_combined_2 = pd.merge(df_combined_1, df_lesion, on=['subject_id', 'study_id', 'dicom_id', 'image_path'], how='left')

In [31]:
print(f"전체 행 수: {len(df_combined_2)}")
df_combined_2.head()

전체 행 수: 243324


,subject_id,study_id,dicom_id,image_path,ViewPosition,cxrtime,Atelectasis,Cardiomegaly,Consolidation,Edema,...,loc_opacity,label_consolidation,mask_consolidation,loc_consolidation,label_edema,mask_edema,loc_edema,label_effusion,mask_effusion,loc_effusion
0,10000032,50414267,02aa804e-bde0afdd-112c0b34-7bc16630-4e384014,files/p10/p10000032/s50414267/02aa804e-bde0afd...,PA,2180-05-06 21:30:14.531,0.0,0.0,0.0,0.0,...,NaN,0.0,None,NaN,0.0,None,NaN,0.0,None,NaN
1,10000032,53189527,2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab,files/p10/p10000032/s53189527/2a2277a9-b0ded15...,PA,2180-06-26 16:55:00.312,0.0,0.0,0.0,0.0,...,NaN,0.0,None,NaN,0.0,None,NaN,0.0,None,NaN
2,10000032,53911762,68b5c4b1-227d0485-9cc38c3f-7b84ab51-4b472714,files/p10/p10000032/s53911762/68b5c4b1-227d048...,AP,2180-07-23 08:05:56.875,0.0,0.0,0.0,0.0,...,NaN,0.0,None,NaN,0.0,None,NaN,0.0,None,NaN
3,10000032,53911762,fffabebf-74fd3a1f-673b6b41-96ec0ac9-2ab69818,files/p10/p10000032/s53911762/fffabebf-74fd3a1...,AP,2180-07-23 08:05:56.875,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10000032,56699142,ea030e7a-2e3b1346-bc518786-7a8fd698-f673b44c,files/p10/p10000032/s56699142/ea030e7a-2e3b134...,AP,2180-08-05 23:44:24.765,0.0,0.0,0.0,0.0,...,NaN,0.0,None,NaN,0.0,None,NaN,0.0,None,NaN


In [32]:
df_reports.head()

,subject_id,study_id,report,txt_path
0,p10992808,s56501444,Mild prominence of the cardiac silhouette is l...,/home/DAHS1/gangmin/my_research/clinical_multi...
1,p10509699,s57504847,"As compared to the previous radiograph, the lu...",/home/DAHS1/gangmin/my_research/clinical_multi...
2,p10509699,s54509601,"Compared to the previous radiograph, the lung ...",/home/DAHS1/gangmin/my_research/clinical_multi...
3,p10509699,s55033134,The lung volumes are low. Moderate cardiomega...,/home/DAHS1/gangmin/my_research/clinical_multi...
4,p10637554,s58966893,"The cardiac, mediastinal and hilar contours ap...",/home/DAHS1/gangmin/my_research/clinical_multi...


In [33]:
df_reports['study_id'] = df_reports['study_id'].str.lstrip('s')
df_reports['subject_id'] = df_reports['subject_id'].str.lstrip('p')

In [34]:
final_cxr_df = df_combined_2.merge(df_reports, on=['subject_id', 'study_id'], how='left').reset_index(drop=True)

In [35]:
print(f"전체 행 수: {len(final_cxr_df)}")
final_cxr_df.head()

전체 행 수: 243324


,subject_id,study_id,dicom_id,image_path,ViewPosition,cxrtime,Atelectasis,Cardiomegaly,Consolidation,Edema,...,mask_consolidation,loc_consolidation,label_edema,mask_edema,loc_edema,label_effusion,mask_effusion,loc_effusion,report,txt_path
0,10000032,50414267,02aa804e-bde0afdd-112c0b34-7bc16630-4e384014,files/p10/p10000032/s50414267/02aa804e-bde0afd...,PA,2180-05-06 21:30:14.531,0.0,0.0,0.0,0.0,...,None,NaN,0.0,None,NaN,0.0,None,NaN,"There is no focal consolidation, pleural effus...",/home/DAHS1/gangmin/my_research/clinical_multi...
1,10000032,53189527,2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab,files/p10/p10000032/s53189527/2a2277a9-b0ded15...,PA,2180-06-26 16:55:00.312,0.0,0.0,0.0,0.0,...,None,NaN,0.0,None,NaN,0.0,None,NaN,"The cardiac, mediastinal and hilar contours ar...",/home/DAHS1/gangmin/my_research/clinical_multi...
2,10000032,53911762,68b5c4b1-227d0485-9cc38c3f-7b84ab51-4b472714,files/p10/p10000032/s53911762/68b5c4b1-227d048...,AP,2180-07-23 08:05:56.875,0.0,0.0,0.0,0.0,...,None,NaN,0.0,None,NaN,0.0,None,NaN,Single frontal view of the chest provided.\n \...,/home/DAHS1/gangmin/my_research/clinical_multi...
3,10000032,53911762,fffabebf-74fd3a1f-673b6b41-96ec0ac9-2ab69818,files/p10/p10000032/s53911762/fffabebf-74fd3a1...,AP,2180-07-23 08:05:56.875,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Single frontal view of the chest provided.\n \...,/home/DAHS1/gangmin/my_research/clinical_multi...
4,10000032,56699142,ea030e7a-2e3b1346-bc518786-7a8fd698-f673b44c,files/p10/p10000032/s56699142/ea030e7a-2e3b134...,AP,2180-08-05 23:44:24.765,0.0,0.0,0.0,0.0,...,None,NaN,0.0,None,NaN,0.0,None,NaN,"The lungs are clear of focal consolidation, pl...",/home/DAHS1/gangmin/my_research/clinical_multi...


In [36]:
final_cxr_df.to_feather('/home/DAHS1/gangmin/my_research/clinical_multimodal_learning/data/final_cxr_df_20260504.ftr')

In [37]:
"""
구성 컬럼
1. 환자 식별자
2. 촬영 시점 및 위치(PA/AP)
3. 원본 CXR, 폐 마스크, 병변 마스크
4. Chexpert 라벨
5. FINDINGS + IMPRESSION 텍스트
"""
final_cxr_df.columns

Index(['subject_id', 'study_id', 'dicom_id', 'image_path', 'ViewPosition',
       'cxrtime', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
       'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 'Lung Opacity',
       'No Finding', 'Pleural Effusion', 'Pleural Other', 'Pneumonia',
       'Pneumothorax', 'Support Devices', 'lung_mask_path',
       'label_cardiomegaly', 'mask_cardiomegaly', 'loc_cardiomegaly',
       'label_pneumonia', 'mask_pneumonia', 'loc_pneumonia',
       'label_atelectasis', 'mask_atelectasis', 'loc_atelectasis',
       'label_opacity', 'mask_opacity', 'loc_opacity', 'label_consolidation',
       'mask_consolidation', 'loc_consolidation', 'label_edema', 'mask_edema',
       'loc_edema', 'label_effusion', 'mask_effusion', 'loc_effusion',
       'report', 'txt_path'],
      dtype='object')

---

# CXR shot in ICU

In [38]:
store_dir = "/home/DAHS1/gangmin/my_research/clinical_multimodal_learning/data/2rd_preprocessed_data/"
key_id_icu = pd.read_feather(store_dir + 'key_id_static.ftr')

dir_path = "/home/DAHS1/gangmin/my_research/clinical_multimodal_learning/data/2rd_preprocessed_data/"
imputed_cxr_df = pd.read_feather(dir_path + '/processed/imputed_cxr_df_20260428.ftr')

In [39]:
key_id_icu = key_id_icu[['subject_id', 'hadm_id', 'stay_id', 'admittime', 'dischtime', 'intime', 'outtime']]

In [40]:
key_id_icu

,subject_id,hadm_id,stay_id,admittime,dischtime,intime,outtime
0,10001884,26184834.0,37510196.0,2131-01-07 20:39:00,2131-01-20 05:15:00,2131-01-11 04:20:05,2131-01-20 08:27:30
1,10002428,23473524.0,35479615.0,2156-05-11 14:49:00,2156-05-22 14:16:00,2156-05-11 14:49:34,2156-05-22 14:16:46
2,10002428,28662225.0,33987268.0,2156-04-12 14:16:00,2156-04-29 16:26:00,2156-04-12 16:24:18,2156-04-17 15:57:08
3,10002428,28662225.0,38875437.0,2156-04-12 14:16:00,2156-04-29 16:26:00,2156-04-19 18:11:19,2156-04-26 18:58:41
4,10003400,20214994.0,32128372.0,2137-02-24 10:00:00,2137-03-19 15:45:00,2137-02-25 23:37:19,2137-03-10 21:29:36
...,...,...,...,...,...,...,...
14141,19998330,24096838.0,33428243.0,2178-11-27 21:51:00,2178-12-01 17:10:00,2178-11-27 22:53:00,2178-11-29 21:29:39
14142,19998843,24842066.0,30988867.0,2187-02-05 09:27:00,2187-02-08 17:28:00,2187-02-05 10:12:00,2187-02-08 18:19:39
14143,19999287,20175828.0,35165301.0,2197-08-03 20:58:00,2197-08-18 15:37:00,2197-08-04 00:02:00,2197-08-08 16:58:17
14144,19999442,26785317.0,32336619.0,2148-11-19 10:00:00,2148-12-04 16:25:00,2148-11-19 14:23:43,2148-11-26 13:12:15


In [41]:
imputed_cxr_df['dicom_id'] = imputed_cxr_df['image_path'].str.split('/').str[-1].str.replace('.jpg', '', regex=False)

In [42]:
test = imputed_cxr_df.copy()
test = test.merge(key_id_icu, on=['hadm_id', 'stay_id'], how='left')

In [43]:
path_split = test['image_path'].str.split('/')

# test['subject_id'] = path_split.str[2]
# test['subject_id'] = test['subject_id'].str.lstrip('p')

test['study_id'] = path_split.str[3]
test['study_id'] = test['study_id'].str.lstrip('s')

In [44]:
test

,hadm_id,stay_id,hour_slot,slot_start,slot_end,cxr_flag,image_path,cxrtime,Atelectasis,Cardiomegaly,...,Pneumothorax,Support Devices,was_missing,dicom_id,subject_id,admittime,dischtime,intime,outtime,study_id
0,26184834,37510196,0,2131-01-11 04:20:05,2131-01-11 05:20:05,1,files/p10/p10001884/s57156853/9fd47edd-0708720...,2131-01-10 12:54:30.328,NaN,NaN,...,NaN,NaN,1.0,9fd47edd-07087209-b901811e-3e9e5f50-f382f611,10001884,2131-01-07 20:39:00,2131-01-20 05:15:00,2131-01-11 04:20:05,2131-01-20 08:27:30,57156853
1,26184834,37510196,1,2131-01-11 05:20:05,2131-01-11 06:20:05,0,None,NaT,NaN,NaN,...,NaN,NaN,0.0,None,10001884,2131-01-07 20:39:00,2131-01-20 05:15:00,2131-01-11 04:20:05,2131-01-20 08:27:30,None
2,26184834,37510196,2,2131-01-11 06:20:05,2131-01-11 07:20:05,0,None,NaT,NaN,NaN,...,NaN,NaN,0.0,None,10001884,2131-01-07 20:39:00,2131-01-20 05:15:00,2131-01-11 04:20:05,2131-01-20 08:27:30,None
3,26184834,37510196,3,2131-01-11 07:20:05,2131-01-11 08:20:05,0,None,NaT,NaN,NaN,...,NaN,NaN,0.0,None,10001884,2131-01-07 20:39:00,2131-01-20 05:15:00,2131-01-11 04:20:05,2131-01-20 08:27:30,None
4,26184834,37510196,4,2131-01-11 08:20:05,2131-01-11 09:20:05,0,None,NaT,NaN,NaN,...,NaN,NaN,0.0,None,10001884,2131-01-07 20:39:00,2131-01-20 05:15:00,2131-01-11 04:20:05,2131-01-20 08:27:30,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1618766,23865745,36195440,42,2145-11-04 16:59:00,2145-11-04 17:59:00,0,None,NaT,NaN,NaN,...,NaN,NaN,0.0,None,19999987,2145-11-02 21:38:00,2145-11-11 12:57:00,2145-11-02 22:59:00,2145-11-04 21:29:30,None
1618767,23865745,36195440,43,2145-11-04 17:59:00,2145-11-04 18:59:00,0,None,NaT,NaN,NaN,...,NaN,NaN,0.0,None,19999987,2145-11-02 21:38:00,2145-11-11 12:57:00,2145-11-02 22:59:00,2145-11-04 21:29:30,None
1618768,23865745,36195440,44,2145-11-04 18:59:00,2145-11-04 19:59:00,0,None,NaT,NaN,NaN,...,NaN,NaN,0.0,None,19999987,2145-11-02 21:38:00,2145-11-11 12:57:00,2145-11-02 22:59:00,2145-11-04 21:29:30,None
1618769,23865745,36195440,45,2145-11-04 19:59:00,2145-11-04 20:59:00,0,None,NaT,NaN,NaN,...,NaN,NaN,0.0,None,19999987,2145-11-02 21:38:00,2145-11-11 12:57:00,2145-11-02 22:59:00,2145-11-04 21:29:30,None


In [45]:
time_processed_cxr_df = test[['subject_id', 'hadm_id', 'stay_id', 'study_id', 'dicom_id', 'slot_start', 'slot_end', 'cxrtime', 'cxr_flag', 'image_path', 'intime', 'outtime']]

In [46]:
time_processed_cxr_df

,subject_id,hadm_id,stay_id,study_id,dicom_id,slot_start,slot_end,cxrtime,cxr_flag,image_path,intime,outtime
0,10001884,26184834,37510196,57156853,9fd47edd-07087209-b901811e-3e9e5f50-f382f611,2131-01-11 04:20:05,2131-01-11 05:20:05,2131-01-10 12:54:30.328,1,files/p10/p10001884/s57156853/9fd47edd-0708720...,2131-01-11 04:20:05,2131-01-20 08:27:30
1,10001884,26184834,37510196,None,None,2131-01-11 05:20:05,2131-01-11 06:20:05,NaT,0,None,2131-01-11 04:20:05,2131-01-20 08:27:30
2,10001884,26184834,37510196,None,None,2131-01-11 06:20:05,2131-01-11 07:20:05,NaT,0,None,2131-01-11 04:20:05,2131-01-20 08:27:30
3,10001884,26184834,37510196,None,None,2131-01-11 07:20:05,2131-01-11 08:20:05,NaT,0,None,2131-01-11 04:20:05,2131-01-20 08:27:30
4,10001884,26184834,37510196,None,None,2131-01-11 08:20:05,2131-01-11 09:20:05,NaT,0,None,2131-01-11 04:20:05,2131-01-20 08:27:30
...,...,...,...,...,...,...,...,...,...,...,...,...
1618766,19999987,23865745,36195440,None,None,2145-11-04 16:59:00,2145-11-04 17:59:00,NaT,0,None,2145-11-02 22:59:00,2145-11-04 21:29:30
1618767,19999987,23865745,36195440,None,None,2145-11-04 17:59:00,2145-11-04 18:59:00,NaT,0,None,2145-11-02 22:59:00,2145-11-04 21:29:30
1618768,19999987,23865745,36195440,None,None,2145-11-04 18:59:00,2145-11-04 19:59:00,NaT,0,None,2145-11-02 22:59:00,2145-11-04 21:29:30
1618769,19999987,23865745,36195440,None,None,2145-11-04 19:59:00,2145-11-04 20:59:00,NaT,0,None,2145-11-02 22:59:00,2145-11-04 21:29:30


In [47]:
"""
1. subject_id, hadm_id, stay_id에서는 결측치가 없어야 함.
2. study_id, dicom_id, cxr_time, image_path는  결측 수가 같아야 함.
3. 
"""
time_processed_cxr_df.isnull().sum()

subject_id          0
hadm_id             0
stay_id             0
study_id      1565474
dicom_id      1565474
slot_start          0
slot_end            0
cxrtime       1565474
cxr_flag            0
image_path    1565474
intime              0
outtime             0
dtype: int64

In [48]:
time_processed_cxr_df['subject_id'] = time_processed_cxr_df['subject_id'].astype(str)

final_processed_cxr_df = time_processed_cxr_df.merge(final_cxr_df, on=['subject_id', 'dicom_id', 'study_id', 'image_path'], how='left')

In [49]:
both_present = final_processed_cxr_df[
    final_processed_cxr_df['cxrtime_x'].notna() & 
    final_processed_cxr_df['cxrtime_y'].notna()
].copy()

mismatch = both_present[both_present['cxrtime_x'] != both_present['cxrtime_y']]

if not mismatch.empty:
    print(f"결측 제외 불일치 행 수: {len(mismatch)}")
    print(mismatch[['subject_id', 'cxrtime_x', 'cxrtime_y']].head())
else:
    print("값이 존재하는 모든 행에서 cxrtime이 완벽하게 일치합니다.")

값이 존재하는 모든 행에서 cxrtime이 완벽하게 일치합니다.


In [50]:
final_processed_cxr_df.drop(columns=['cxrtime_y'], inplace=True)
final_processed_cxr_df = final_processed_cxr_df.rename(columns={'cxrtime_x': 'cxrtime'})

In [51]:
final_processed_cxr_df

,subject_id,hadm_id,stay_id,study_id,dicom_id,slot_start,slot_end,cxrtime,cxr_flag,image_path,...,mask_consolidation,loc_consolidation,label_edema,mask_edema,loc_edema,label_effusion,mask_effusion,loc_effusion,report,txt_path
0,10001884,26184834,37510196,57156853,9fd47edd-07087209-b901811e-3e9e5f50-f382f611,2131-01-11 04:20:05,2131-01-11 05:20:05,2131-01-10 12:54:30.328,1,files/p10/p10001884/s57156853/9fd47edd-0708720...,...,None,NaN,0.0,None,NaN,0.0,None,NaN,"In comparison to ___ chest radiograph, pulmona...",/home/DAHS1/gangmin/my_research/clinical_multi...
1,10001884,26184834,37510196,None,None,2131-01-11 05:20:05,2131-01-11 06:20:05,NaT,0,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10001884,26184834,37510196,None,None,2131-01-11 06:20:05,2131-01-11 07:20:05,NaT,0,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10001884,26184834,37510196,None,None,2131-01-11 07:20:05,2131-01-11 08:20:05,NaT,0,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10001884,26184834,37510196,None,None,2131-01-11 08:20:05,2131-01-11 09:20:05,NaT,0,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1618766,19999987,23865745,36195440,None,None,2145-11-04 16:59:00,2145-11-04 17:59:00,NaT,0,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1618767,19999987,23865745,36195440,None,None,2145-11-04 17:59:00,2145-11-04 18:59:00,NaT,0,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1618768,19999987,23865745,36195440,None,None,2145-11-04 18:59:00,2145-11-04 19:59:00,NaT,0,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1618769,19999987,23865745,36195440,None,None,2145-11-04 19:59:00,2145-11-04 20:59:00,NaT,0,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [53]:
final_processed_cxr_df.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'study_id', 'dicom_id',
       'slot_start', 'slot_end', 'cxrtime', 'cxr_flag', 'image_path', 'intime',
       'outtime', 'ViewPosition', 'Atelectasis', 'Cardiomegaly',
       'Consolidation', 'Edema', 'Enlarged Cardiomediastinum', 'Fracture',
       'Lung Lesion', 'Lung Opacity', 'No Finding', 'Pleural Effusion',
       'Pleural Other', 'Pneumonia', 'Pneumothorax', 'Support Devices',
       'lung_mask_path', 'label_cardiomegaly', 'mask_cardiomegaly',
       'loc_cardiomegaly', 'label_pneumonia', 'mask_pneumonia',
       'loc_pneumonia', 'label_atelectasis', 'mask_atelectasis',
       'loc_atelectasis', 'label_opacity', 'mask_opacity', 'loc_opacity',
       'label_consolidation', 'mask_consolidation', 'loc_consolidation',
       'label_edema', 'mask_edema', 'loc_edema', 'label_effusion',
       'mask_effusion', 'loc_effusion', 'report', 'txt_path'],
      dtype='object')

In [54]:
final_processed_cxr_df.to_feather('/home/DAHS1/gangmin/my_research/clinical_multimodal_learning/data/final_cxr_df_in_icu_20260504.ftr')

In [ ]:
# chexpert 라벨과 ils 데이터셋에서 만든 ex) label_effusion이 동일한지 확인해야 함.
